# Chapter 2: Data Cleaning & Preprocessing

Machine learning algorithms are mathematical equations. They cannot read raw text, they get confused by missing values (`NaN`), and they can easily be skewed if one column contains tiny numbers (like age) while another contains huge numbers (like salary). **Data preprocessing** is the process of washing, slicing, and preparing your raw data before feeding it to a model.

### High-Level Intuition & Analogy
Imagine preparing raw vegetables for cooking: you wash off the dirt, peel off the bad spots, and chop everything into uniform sizes. In data cleaning:
* **Missing values** are the bad spots - we must either throw them away (drop rows) or replace them with a sensible value like the average (impute them).
* **Categorical variables** (like 'City' or 'Size') are text tags - we must translate them into numbers (encode them) so the computer can compute with them.
* **Feature Scaling** is making sure all features share a standard range - so columns with larger numerical scales don't dominate the mathematical equations.

### Core Concepts & Step-by-Step Execution
1. **Missing Data Types**:
   * **MCAR (Missing Completely at Random)**: Missingness is purely random (e.g., equipment glitch).
   * **MAR (Missing at Random)**: Missingness depends on other observed variables (e.g., older patients omitting weight).
   * **MNAR (Missing Not at Random)**: Missingness depends on the unobserved value itself (e.g., high-income individuals hiding income).
2. **Encoding Categorical Columns**:
   * **Ordinal Encoding**: Used when categories have a natural sequence (e.g., Low=0, Medium=1, High=2). We map them directly to integer series.
   * **One-Hot Encoding**: Used when categories have no logical sequence (e.g., Red, Blue, Green). We create dummy binary columns for each category (e.g., `Color_Red`, `Color_Blue`), so the model doesn't assume that Blue is "greater than" Red.
3. **Feature Scaling**:
   * **Min-Max Scaling**: Squashes values into a $[0, 1]$ range. Formula: $x_{scaled} = \frac{x - x_{min}}{x_{max} - x_{min}}$.
   * **Standardization (Z-score)**: Centers data around 0 with a standard deviation of 1. Formula: $z = \frac{x - \mu}{\sigma}$ (where $\mu$ is the mean and $\sigma$ is the standard deviation).

### Mathematical Foundations (Simplified)
* **Standardization Formula**: $z = \frac{x - \mu}{\sigma}$. If a data point's value is equal to the mean ($\mu$), its Z-score becomes 0. If it is one standard deviation above the mean, its Z-score is 1. This keeps the distribution shape intact while aligning all features.
* **Dummy Variable Trap**: When one-hot encoding, if you include dummy columns for all $N$ categories, they will be perfectly collinear (since their sum is always 1). This breaks linear models. We prevent this by dropping the first category, leaving $N-1$ columns.

### Pros and Cons
* **Pros**: Improves model convergence speed and accuracy, handles mixed data types, and prevents mathematical overflow in gradient descent.
* **Cons**: Imputation can introduce bias if missingness is not random, and scaling can mask outliers if not handled carefully.
* **Robust Alternatives**: For datasets with extreme outliers, use `RobustScaler`, which scales features using the median and Interquartile Range (IQR) rather than the mean and standard deviation.

### Pro-Tips & Gotchas
* **Data Leakage**: Never fit preprocessing scalers on the entire dataset! Always split your data into training and test sets first, fit the scaler *only* on the training set, and then transform both sets. Otherwise, information from the test set "leaks" into the training phase.

### Programming Guide & Key Functions
Here is a comprehensive guide to preprocessing functions in Scikit-Learn and Pandas:

```python
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler, RobustScaler

# --- Missing Value Imputation ---
df = pd.DataFrame({'Age': [25, np.nan, 35, 40], 'Salary': [50000, 60000, np.nan, 80000]})

imputer = SimpleImputer(strategy='median')  # Impute using median
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

knn_imputer = KNNImputer(n_neighbors=2)  # Impute using K-Nearest Neighbors
df_knn = pd.DataFrame(knn_imputer.fit_transform(df), columns=df.columns)

# --- Categorical Encoding ---
df_cat = pd.DataFrame({'Color': ['Red', 'Blue', 'Green', 'Red'], 'Size': ['S', 'M', 'L', 'M']})

# Ordinal Encoding (ordered categories)
ord_enc = OrdinalEncoder(categories=[['S', 'M', 'L']])
df_cat['Size_Encoded'] = ord_enc.fit_transform(df_cat[['Size']])

# One-Hot Encoding (unordered categories)
ohe = OneHotEncoder(drop='first', sparse_output=False)  # Drop first to avoid dummy variable trap
ohe_features = ohe.fit_transform(df_cat[['Color']])

# --- Feature Scaling ---
data = [[10, 10000], [20, 20000], [30, 30000], [100, 100000]]  # Last row is an outlier

# MinMaxScaler scales to [0,1]
mms = MinMaxScaler()
data_mms = mms.fit_transform(data)

# StandardScaler centers around 0, scales std to 1
std = StandardScaler()
data_std = std.fit_transform(data)

# RobustScaler handles outliers safely using IQR
rs = RobustScaler()
data_rs = rs.fit_transform(data)
```

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler

# Example of imputation
s = pd.Series([1, np.nan, 3, np.nan, 5])
print("Imputed with mean:", s.fillna(s.mean()).tolist())

# Example of One-Hot Encoding
df_cat = pd.DataFrame({"Color": ["Red", "Blue", "Red"]})
onehot = pd.get_dummies(df_cat, columns=["Color"])
print("\nOne-hot Encoded:")
print(onehot)

---
## Exercises

Now it is your turn! Run the verification code cell after each exercise to check your answers.

### Exercise 1: Impute Constant Value (Easy)
Given a Pandas Series `s_easy` with missing values, impute (fill) all NaNs with a constant value of `0.0`. Store the result in `s_filled`.

In [ ]:
import pandas as pd
import numpy as np

s_easy = pd.Series([1.5, np.nan, 2.5, np.nan, 3.5])
# TODO: Fill NaNs with 0.0
s_filled = ____
print(s_filled)

In [ ]:
from learntools import ch2

ch2.step_1.check(s_filled)
# ch2.step_1.hint()# ch2.step_1.solution()

### Exercise 2: Drop Missing Values (Easy)
Given a Pandas DataFrame `df_miss`, drop all rows containing any missing values and store the result in `df_clean`.

In [ ]:
df_miss = pd.DataFrame({"A": [1, np.nan, 3], "B": [4, 5, np.nan]})
# TODO: Drop rows with missing values
df_clean = ____
print(df_clean)

In [ ]:
ch2.step_2.check(df_clean)
# ch2.step_2.hint()# ch2.step_2.solution()

### Exercise 3: Lowercase Categorical Strings (Easy)
Given a Pandas Series `s_str`, convert all string values to lowercase. Store the result in `s_lower`.

In [ ]:
s_str = pd.Series(["RED", "Blue", "GREEN"])
# TODO: Convert to lowercase
s_lower = ____
print(s_lower)

In [ ]:
ch2.step_3.check(s_lower)
# ch2.step_3.hint()# ch2.step_3.solution()

### Exercise 4: Fill Missing Values with Median (Easy)
Fill any missing values (NaNs) in the Pandas Series `s_to_fill` with its median value and store the result in `s_median_filled`.

In [ ]:
s_to_fill = pd.Series([1.0, np.nan, 3.0, 4.0, 5.0])
# TODO: Fill missing values with median
s_median_filled = ____
print(s_median_filled)

In [ ]:
ch2.step_7.check(s_median_filled)
# ch2.step_7.hint()# ch2.step_7.solution()

### Exercise 5: Scale Feature to [0, 1] range (Easy/Medium)
Scale the 1D NumPy array `arr_to_scale` to range `[0, 1]` using min-max scaling and store the result in `arr_scaled_01`.

In [ ]:
arr_to_scale = np.array([10.0, 15.0, 20.0])
# TODO: Perform min-max scaling to range [0, 1]
arr_scaled_01 = ____
print(arr_scaled_01)

In [ ]:
ch2.step_8.check(arr_scaled_01)
# ch2.step_8.hint()# ch2.step_8.solution()

### Exercise 6: Missing Value Imputation (Medium)
Impute the missing values (`NaN`) in the series `s` below using the **mean** value of the series. Store the output in a variable `s_imputed`.

In [ ]:
s = pd.Series([10.0, np.nan, 20.0, 30.0, np.nan, 40.0])

# TODO: Impute missing values with mean
s_imputed = ____

print(s_imputed)

In [ ]:
ch2.step_4.check(s_imputed)
# ch2.step_4.hint()# ch2.step_4.solution()

### Exercise 7: Ordinal & One-Hot Encoding (Medium)
Given a DataFrame `df` containing a categorical column `'Size'` with values `['Small', 'Medium', 'Large', 'Medium', 'Small']` and a `'Color'` column.
1. Perform **Ordinal Encoding** on the `'Size'` column. Map `'Small'` to `0`, `'Medium'` to `1`, and `'Large'` to `2`. Store this in a new column `'Size_Encoded'`.
2. Perform **One-Hot Encoding** on the `'Color'` column. Use Pandas `pd.get_dummies()` and ensure the resulting columns prefix is `'Color'`. Keep all original columns (except `'Color'`, which get_dummies replaces automatically).
Store your final DataFrame as `encoded_df`.

In [ ]:
df = pd.DataFrame(
    {
        "Size": ["Small", "Medium", "Large", "Medium", "Small"],
        "Color": ["Red", "Blue", "Red", "Green", "Blue"],
    }
)

# TODO: Perform Ordinal encoding on 'Size' (0, 1, 2)
# and One-Hot encoding on 'Color'.
encoded_df = ____

print(encoded_df)

In [ ]:
ch2.step_5.check(encoded_df)
# ch2.step_5.hint()# ch2.step_5.solution()

### Exercise 8: Custom Scaling with Percentile Clipping (Hard)
Outliers can severely affect machine learning models. Write a function `custom_scale(arr, min_val, max_val)` that:
1. Identifies the **10th percentile** ($p_{10}$) and **90th percentile** ($p_{90}$) of the 1D input array `arr`.
2. **Clips** any values below $p_{10}$ to $p_{10}$, and any values above $p_{90}$ to $p_{90}$.
3. **Scales** the clipped array linearly into the range $[min\_val, max\_val]$.

**Formulas**:
Min-max scaling to range $[a, b]$:
$$x_{scaled} = \frac{x - x_{min}}{x_{max} - x_{min}} \cdot (b - a) + a$$

In [ ]:
def custom_scale(arr, min_val, max_val):
    # TODO: Implement 10/90 percentile clipping and min-max scaling to [min_val, max_val]
    pass


# Test custom scaling
arr = np.array([-100, 1, 2, 3, 4, 5, 6, 7, 8, 100])
print("Scaled Array:", custom_scale(arr, 0.0, 1.0))

In [ ]:
ch2.step_6.check(custom_scale)
# ch2.step_6.hint()# ch2.step_6.solution()